# SR3 — 확산 모델로 초해상

앞의 일곱 모델은 저해상도를 넣으면 고해상도가 한 번에 나온다. SR3 는 다르다.
**순전한 잡음에서 시작해 조금씩 걷어내며** 그림을 만든다. 저해상도 영상은 "이렇게 생긴
것을 만들라" 는 조건으로 매 스텝 들어간다.

```
잡음  →  조금 걷어냄  →  조금 더  →  …  →  결과
          ↑ 매 스텝 저해상도 영상을 조건으로 본다
```

**미리 알아둘 것.** 이 페이지의 가중치는 **20,100 iteration** 에서 멈춘 것이다.
원 논문은 100만 iteration 을 돌린다 — **2% 만 학습한 셈**이다. 그래서 결과가 bicubic 보다
못하다. 이 페이지는 그 사실을 감추지 않고, 왜 그런지와 무엇을 조절할 수 있는지를 본다.

결과는 미리 뽑아 저장소에 올려둔 것을 불러온다. **GPU 가 필요 없다.**

## 1. 준비

In [ ]:
import sys, json, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
urllib.request.urlretrieve(f'{BASE}/lib/sr_utils.py', 'sr_utils.py')
sys.modules.pop('sr_utils', None)
from sr_utils import *
import pandas as pd

RES = f'{BASE}/results/sr3'
V = 's1'          # 내려받은 파일 캐시 폴더
META = json.load(open(fetch(f'{RES}/meta.json', f'{V}/meta.json')))
PATCHES = list(META['patches'])
CENTER, SIZE = (67, 370), 90       # 다른 페이지와 같은 확대 자리


def load(name):
    return imageio.imread(fetch(f'{RES}/{name}.png', f'{V}/{name}.png'))


print(f"체크포인트 {META['ckpt']}   되돌리는 스텝 {META['steps']}   색보정 {META['mode']}")
print('패치:', ', '.join(PATCHES))

## 2. 잡음에서 그림이 나오는 과정

되돌리기를 시작할 때는 **순전한 잡음**이다. 스텝을 밟을수록 저해상도 영상이 조건으로
끌어당겨 형태가 잡힌다. 아래는 그 과정을 몇 지점에서 잘라 본 것이다.

In [ ]:
stem = PATCHES[0]
keep = META['keep']
fig, ax = plt.subplots(1, len(keep) + 1, figsize=(2.4 * (len(keep) + 1), 3.0))
for j, p in enumerate(keep):
    ax[j].imshow(imageio.imread(fetch(f'{RES}/steps/{stem}_t{p:03d}.png',
                                      f'{V}/steps/{stem}_t{p:03d}.png')))
    ax[j].set_title(f'{p}%', fontsize=12)
ax[-1].imshow(load(f'{stem}_HR')); ax[-1].set_title('Target HR', fontsize=12)
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.suptitle('reverse process — noise to image', fontsize=13)
plt.tight_layout(); plt.show()

**0% 는 완전한 잡음이다.** 25% 쯤에서 큰 덩어리가 잡히고, 50% 를 넘으면 도로와 블록이
보이기 시작한다. 마지막까지 가도 정답만큼 또렷해지지는 않는데, 학습이 모자란 탓이다.

## 3. 결과 비교

같은 자리를 다른 페이지와 똑같이 확대해서 본다.

In [ ]:
for stem in PATCHES:
    zoom([('Original LR', load(f'{stem}_LR')),
          ('Bicubic', load(f'{stem}_Bicubic')),
          ('SR3', load(f'{stem}_SR3')),
          ('Target HR', load(f'{stem}_HR'))],
         center=CENTER, size=SIZE, title=stem)

In [ ]:
print(f'{"":34s}{"PSNR":>9s}{"SSIM":>9s}')
for stem, v in META['patches'].items():
    print(f'{stem:26s} bicubic {v["bicubic"][0]:9.2f}{v["bicubic"][1]:9.4f}')
    print(f'{"":26s} SR3     {v["sr3"][0]:9.2f}{v["sr3"][1]:9.4f}')
print('\nSR3 가 bicubic 보다 낮다. 확대 그림을 보면 형태는 따라가는데 질감이 거칠다.')

## 4. 무엇을 조절할 수 있나

확산 모델은 학습이 끝난 뒤에도 **추론 때 고를 것이 남아 있다.** 다른 모델에는 없는 특징이다.

- **되돌리는 스텝 수** — 많이 밟을수록 곱게 만들지만 그만큼 느리다
- **잡음 일정(schedule)** — 잡음을 얼마나 빨리 넣고 뺄지
- **색 보정** — 되돌리는 동안 색이 표류하는 것을 매 스텝 조건 영상에 맞춰 잡아준다

스텝 수와 일정을 바꿔가며 잰 표가 있다. 실제 Sentinel-2 LR 입력으로 잰 값이라
위 3절(합성 입력)과 잣대가 다르다 — 표 안에서만 비교한다.

In [ ]:
sw = pd.read_csv(fetch(f'{RES}/sweep.csv', f'{V}/sweep.csv'))
st = sw[sw.sweep == 'steps']

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(st.n_timestep, st.PSNR, 'o-', color='#4f7fa8')
ax[0].set_xscale('log'); ax[0].set_xlabel('steps'); ax[0].set_ylabel('PSNR')
ax[1].plot(st.n_timestep, st.SSIM, 'o-', color='#c96a5b')
ax[1].set_xscale('log'); ax[1].set_xlabel('steps'); ax[1].set_ylabel('SSIM')
ax[2].plot(st.n_timestep, st.sec, 'o-', color='#4f9f6f')
ax[2].set_xscale('log'); ax[2].set_xlabel('steps'); ax[2].set_ylabel('seconds')
for a in ax:
    a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(st[['n_timestep', 'PSNR', 'SSIM', 'sec']].to_string(index=False))

**PSNR 은 100 스텝에서 꺾이는데 SSIM 은 계속 오른다.** 스텝을 더 밟으면 질감이 살아나지만
화소값은 오히려 정답에서 멀어진다는 뜻이다 — 4·5번 페이지의 GAN 계열에서 본 것과 같은
지각-왜곡 트레이드오프다.

시간은 스텝 수에 정비례한다. 400 스텝이면 한 장에 43초다. 앞의 일곱 모델은 0.1초도 안 걸린다.

In [ ]:
sc = sw[sw.sweep == 'schedule'][['schedule', 'PSNR', 'SSIM']]
le = sw[sw.sweep == 'linear_end'][['linear_end', 'PSNR', 'SSIM']]
print('잡음 일정을 바꾸면\n'); print(sc.to_string(index=False))
print('\n잡음 세기(linear_end)를 바꾸면\n'); print(le.to_string(index=False))
print(f"\n일정을 바꾸면 PSNR 은 {sc.PSNR.max()-sc.PSNR.min():.2f} dB 안에서만 움직인다.")
print(f"잡음 세기는 {le.PSNR.max()-le.PSNR.min():.2f} dB 로 폭이 크지만, 가장 좋은 값도 "
      f"{le.PSNR.max():.2f} 다.")
print('같은 입력에서 bicubic 이 15.08 이므로, 추론 설정으로 메울 수 있는 격차가 아니다.')

## 5. 정리

**확산 모델은 만드는 방식이 다르다.** 한 번에 답을 내지 않고 잡음에서 시작해 여러 번
되돌린다. 그래서 추론이 느리고(400 스텝에 43초), 학습 뒤에도 스텝 수·일정·색보정 같은
선택지가 남는다.

**이 페이지의 가중치는 2% 만 학습한 것이다.** 20,100 / 1,000,000 iteration. 그래서 bicubic
보다 낮다. 추론 설정을 아무리 골라도 4절 표에서 가장 좋은 값이 10.32 인데 같은 입력의
bicubic 이 15.08 이다. **부족한 것은 학습량이지 설정이 아니다.**

**그래도 SSIM 곡선은 볼 만하다.** 스텝을 늘릴수록 계속 오른다. 화소를 맞히는 능력과 질감을
만드는 능력이 따로 논다는 것을, 같은 모델 안에서 스텝 하나로 조절해 보인 셈이다.